# 🔍 UYDS SSS Veri Ayıklama Pipeline'
## A100 GPU · Çoklu Embedding Modeli · 13 Hata Kategorisi

**Amaç:** `UYDS_SSS_Listesi.xlsx` içindeki hatalı/bozuk/tekrarlı kayıtları otomatik tespit edip ayıklamak.

**Çıktılar:**
- `temiz_veri.xlsx` — Ayıklanmış, sadece geçerli satırlar
- `hatali_kayitlar.xlsx` — Hata sınıfları ve önerilen aksiyonlarla birlikte
- `ozet.json` — İstatistiksel özet

---

## ⚙️ Hücre 0: Ortam Kurulumu

In [ ]:
# ============================================================
# HÜCRE 0: Google Drive bağlama + pip paketleri
# ============================================================
import sys, subprocess, os, warnings
warnings.filterwarnings("ignore")

# --- Google Drive bağlama ---
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# --- GPU kontrolü ---
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
if gpu_info:
    print(f"✅ GPU: {gpu_info[0]}")
else:
    print("❌ GPU bulunamadı! Runtime → Change runtime type → A100 seçin.")
    sys.exit(1)

# --- Pip kurulumları ---
print("📦 Paketler yükleniyor...")
!pip install -qU \
    sentence-transformers==3.4.1 \
    torch==2.5.1+cu121 \
    transformers==4.47.1 \
    pandas==2.2.3 \
    openpyxl==3.1.5 \
    scikit-learn==1.6.0 \
    numpy==1.26.4 \
    lingua-language-detector==2.0.2 \
    hunspell==0.5.5 \
    tqdm==4.67.1 \
    accelerate==1.2.1 \
    bitsandbytes==0.45.0
print("✅ Paketler yüklendi.")

# --- Hunspell Türkçe sözlük ---
if not os.path.exists("/usr/share/hunspell/tr_TR.dic"):
    !apt-get install -y hunspell-tr > /dev/null 2>&1
    print("✅ Türkçe hunspell sözlüğü kuruldu.")

# --- Çalışma dizini ---
# Varsayılan: Bu notebook'un bulunduğu Google Drive klasörü
# Excel dosyasını da aynı klasöre koyun.
# Notebook yolunu otomatik bulmaya çalışalım:
import glob as _glob
# Drive'da notebook'un olduğu yeri bul
CALISMA_DIZINI = "/content/drive/MyDrive"
# Eğer google_collab_egitim klasörünü bulabilirsek onu kullanalım:
aday_dizinler = _glob.glob("/content/drive/MyDrive/**/google_collab_egitim", recursive=True)
if aday_dizinler:
    CALISMA_DIZINI = aday_dizinler[0]
    print(f"📁 Çalışma dizini: {CALISMA_DIZINI}")
else:
    print(f"⚠️ 'google_collab_egitim' bulunamadı. Varsayılan dizin: {CALISMA_DIZINI}")
    print("   Google Drive > MyDrive içine 'google_collab_egitim' klasörünü yükleyip tekrar çalıştırın.")

# ⚠️ BU SATIRI DEĞİŞTİRİN: Excel dosyanızın tam adı
EXCEL_DOSYASI = os.path.join(CALISMA_DIZINI, "UYDS_SSS_Listesi.xlsx")
# Eğer dosya yoksa, alternatif yolları dene:
if not os.path.exists(EXCEL_DOSYASI):
    alternatifler = _glob.glob(os.path.join(CALISMA_DIZINI, "*.xlsx"))
    if alternatifler:
        EXCEL_DOSYASI = alternatifler[0]
        print(f"📄 Bulunan Excel: {EXCEL_DOSYASI}")
    else:
        print(f"❌ Excel dosyası bulunamadı! Lütfen {CALISMA_DIZINI} içine .xlsx dosyasını koyun.")

CIKTI_DIZINI = os.path.join(CALISMA_DIZINI, "ciktilar")
os.makedirs(CIKTI_DIZINI, exist_ok=True)

print(f"📄 Excel: {EXCEL_DOSYASI}")
print(f"📁 Çıktı: {CIKTI_DIZINI}")
print("\n✅ Hücre 0 tamam.")

## 📖 Hücre 1: Excel'i Oku + İlk Bakış

In [ ]:
# ============================================================
# HÜCRE 1: Veriyi pandas'a yükle
# ============================================================
import pandas as pd
import numpy as np
import json
from datetime import datetime

# --- Oku ---
print(f"📖 Okunuyor: {EXCEL_DOSYASI}")
veri = pd.read_excel(EXCEL_DOSYASI, sheet_name=0, engine="openpyxl", dtype=object)
print(f"✅ Okundu: {len(veri)} satır × {len(veri.columns)} sütun")
print(f"   Sütunlar: {list(veri.columns)}")
# İlk 3 satırı göster
veri.head(3)

## 🔤 Hücre 2: Sütun İsimlerini Normalleştir

In [ ]:
# ============================================================
# HÜCRE 2: UYDS formatına göre sütunları eşleştir
# ============================================================
import unicodedata

def adi_normalize_et(ad: str) -> str:
    """Türkçe karakterleri koruyarak normalize et."""
    if not isinstance(ad, str):
        return ""
    ad = unicodedata.normalize("NFKC", ad)
    return ad.strip().lower().replace("ı", "i").replace("ğ", "g").replace("ü", "u").replace("ş", "s").replace("ö", "o").replace("ç", "c")

# Mevcut sütunları normalize et
mevcut_kolonlar = {adi_normalize_et(k): k for k in veri.columns}
print(f"Mevcut sütunlar (normalleştirilmiş): {list(mevcut_kolonlar.keys())}")

# Hedef sütun eşleme
# UYDS formatı
HEDEF_KOLONLAR = {
    "modul": ["modul"],
    "konu": ["konu"],
    "aciklama": ["aciklama"],
    "yapilacak": ["yapilacak"],
    "yapilan_islem": ["yapilan islem", "yapilan_islem"],
    "madde_no": ["madde no", "madde_no", "harici kod"],
    "madde_tarihi": ["madde tarihi", "madde_tarihi"],
    "musteri": ["musteri"],
}
# Standart format (yedek)
STANDART_KOLONLAR = {
    "kategori": ["kategori"],
    "soru": ["soru"],
    "cevap": ["cevap"],
    "kaynak": ["kaynak dokuman", "kaynak belge", "kaynak"],
}

# Hangisi eşleşiyor?
def kolonlari_esle(hedef_kolonlar: dict) -> dict:
    esleme = {}
    for alan, adaylar in hedef_kolonlar.items():
        for aday in adaylar:
            norm_aday = adi_normalize_et(aday)
            if norm_aday in mevcut_kolonlar:
                esleme[alan] = mevcut_kolonlar[norm_aday]
                break
    return esleme

uyds_esleme = kolonlari_esle(HEDEF_KOLONLAR)
standart_esleme = kolonlari_esle(STANDART_KOLONLAR)

if len(uyds_esleme) >= 4:  # En az 4 UYDS sütunu bulduysa
    format_turu = "uyds"
    esleme = uyds_esleme
    print(f"✅ UYDS formatı algılandı. Eşlenen: {list(uyds_esleme.keys())}")
elif len(standart_esleme) >= 3:
    format_turu = "standart"
    esleme = standart_esleme
    print(f"✅ Standart format algılandı. Eşlenen: {list(standart_esleme.keys())}")
else:
    raise ValueError(f"Excel formatı tanınamadı. Mevcut sütunlar: {list(mevcut_kolonlar.keys())}")

print(f"   Format: {format_turu}")
print(f"   Eşleme: {json.dumps(esleme, ensure_ascii=False, indent=2)}")

## 🏗️ Hücre 3: Soru/Cevap/Kategori Alanlarını Oluştur

In [ ]:
# ============================================================
# HÜCRE 3: UYDS satırlarından soru/cevap/kategori çıkar
# ============================================================

def metne_donustur(deger) -> str:
    """None/NaN/float değerleri güvenli string'e çevir."""
    if deger is None or (isinstance(deger, float) and pd.isna(deger)):
        return ""
    if isinstance(deger, (int, float)):
        return str(int(deger)) if deger == int(deger) else str(deger)
    return str(deger).strip()

if format_turu == "uyds":
    veri["_modul"] = veri[esleme["modul"]].apply(metne_donustur) if "modul" in esleme else ""
    veri["_konu"] = veri[esleme["konu"]].apply(metne_donustur) if "konu" in esleme else ""
    veri["_aciklama"] = veri[esleme["aciklama"]].apply(metne_donustur) if "aciklama" in esleme else ""
    veri["_yapilacak"] = veri[esleme["yapilacak"]].apply(metne_donustur) if "yapilacak" in esleme else ""
    veri["_yapilan_islem"] = veri[esleme["yapilan_islem"]].apply(metne_donustur) if "yapilan_islem" in esleme else ""
    veri["_madde_no"] = veri[esleme["madde_no"]].apply(metne_donustur) if "madde_no" in esleme else ""
    veri["_madde_tarihi"] = veri[esleme["madde_tarihi"]].apply(metne_donustur) if "madde_tarihi" in esleme else ""
    veri["_musteri"] = veri[esleme["musteri"]].apply(metne_donustur) if "musteri" in esleme else ""
    
    # Kategori = modul
    veri["_kategori"] = veri["_modul"].apply(lambda x: x if x else "Belirtilmemiş")
    # Soru = konu (yoksa aciklama, yoksa yapilacak)
    veri["_soru"] = veri.apply(
        lambda r: r["_konu"] or r["_aciklama"] or r["_yapilacak"] or r["_yapilan_islem"],
        axis=1
    )
    # Cevap = yapilan_islem (yoksa yapilacak, yoksa aciklama)
    veri["_cevap"] = veri.apply(
        lambda r: r["_yapilan_islem"] or r["_yapilacak"] or r["_aciklama"],
        axis=1
    )
else:
    veri["_kategori"] = veri[esleme["kategori"]].apply(metne_donustur)
    veri["_soru"] = veri[esleme["soru"]].apply(metne_donustur)
    veri["_cevap"] = veri[esleme["cevap"]].apply(metne_donustur)
    veri["_modul"] = veri["_kategori"]
    veri["_yapilan_islem"] = veri["_cevap"]

print(f"📊 Kategori: {veri['_kategori'].nunique()} benzersiz")
print(f"📊 Toplam: {len(veri)} satır")
print(f"\n📋 İlk 5 satır (soru/cevap):")
veri[["_kategori", "_soru", "_cevap"]].head()

## 🔍 Hücre 4: Yapısal Hata Tespiti (Kategori 5-12)

Bu hücrede embedding gerektirmeyen, hızlı regex/metrik kontrolleri yapılır.

In [ ]:
# ============================================================
# HÜCRE 4: Yapısal kontroller (embedding'siz)
# ============================================================
import re
from collections import Counter

# --- Yardımcı fonksiyonlar ---
TURKCE_KARAKTERLER = set("abcçdefgğhıijklmnoöprsştuüvyzABCÇDEFGĞHIİJKLMNOÖPRSŞTUÜVYZ")
SADECE_SEMBOL_RE = re.compile(r"^[\d\W_]+$")

def token_sayisi(metin: str) -> int:
    return len(metin.split()) if metin else 0

def turkce_karakter_orani(metin: str) -> float:
    if not metin:
        return 0.0
    harfler = [c for c in metin if c.isalpha()]
    if not harfler:
        return 0.0
    turkce_harfler = [c for c in harfler if c in TURKCE_KARAKTERLER]
    return len(turkce_harfler) / len(harfler)

def sadece_sembol_mu(metin: str) -> bool:
    if not metin:
        return False
    return bool(SADECE_SEMBOL_RE.match(metin))

def kelime_cesitliligi(metin: str) -> float:
    """Type-Token Ratio (TTR)."""
    kelimeler = metin.lower().split()
    if not kelimeler:
        return 0.0
    return len(set(kelimeler)) / len(kelimeler)

# --- Tespit edilecek hata sınıfları ---
KISA_SORU_ESIK = 3       # Token sayısı < 3 → anlamsız olabilir
UZUN_SORU_ESIK = 150     # Token sayısı > 150 → paragraf sorusu

hata_kolonlari = []

# Kategori 5: Boş/eksik alan
bos_kategori = veri["_kategori"] == ""
bos_soru = veri["_soru"] == ""
bos_cevap = veri["_cevap"] == ""

veri["_hata_bos_kategori"] = bos_kategori
veri["_hata_bos_soru"] = bos_soru
veri["_hata_bos_cevap"] = bos_cevap
hata_kolonlari.extend(["_hata_bos_kategori", "_hata_bos_soru", "_hata_bos_cevap"])

# Kategori 5b: Çok kısa soru
veri["_soru_token"] = veri["_soru"].apply(token_sayisi)
veri["_hata_kisa_soru"] = (veri["_soru_token"] > 0) & (veri["_soru_token"] < KISA_SORU_ESIK)
hata_kolonlari.append("_hata_kisa_soru")

# Kategori 6: Çok uzun soru
veri["_hata_uzun_soru"] = veri["_soru_token"] > UZUN_SORU_ESIK
hata_kolonlari.append("_hata_uzun_soru")

# Kategori 7: Sadece sayı/sembol
veri["_hata_sadece_sembol"] = veri["_soru"].apply(sadece_sembol_mu)
hata_kolonlari.append("_hata_sadece_sembol")

# Kategori 12: Bilgi yoğunluğu düşük (TTR < 0.3)
veri["_soru_ttr"] = veri["_soru"].apply(kelime_cesitliligi)
veri["_hata_dusuk_bilgi"] = (veri["_soru_token"] > 5) & (veri["_soru_ttr"] < 0.3)
hata_kolonlari.append("_hata_dusuk_bilgi")

# --- Özet ---
yapisal_hatalar = veri[hata_kolonlari].any(axis=1)
print(f"🔍 Yapısal kontrol sonuçları:")
for kolon in hata_kolonlari:
    sayi = veri[kolon].sum()
    if sayi > 0:
        print(f"   {kolon}: {sayi} satır")
print(f"\n📊 Yapısal hatası olan toplam satır: {yapisal_hatalar.sum()} / {len(veri)}")
print(f"✅ Hücre 4 tamam.")

## 🌐 Hücre 5: Dil Tespiti (Kategori 8)

Türkçe olmayan soruları tespit eder.

In [ ]:
# ============================================================
# HÜCRE 5: Dil tespiti
# ============================================================
from lingua import Language, LanguageDetectorBuilder

# Sadece ilgilendiğimiz dilleri tespit et
languages = [Language.TURKISH, Language.ENGLISH, Language.ARABIC, 
             Language.RUSSIAN, Language.FRENCH, Language.GERMAN]
detector = LanguageDetectorBuilder.from_languages(*languages).build()

def dili_tespit_et(metin: str) -> str:
    """Metnin dilini tespit et. Boş metin → 'bos'."""
    if not metin or not metin.strip():
        return "bos"
    try:
        # lingua çok kısa metinlerde hata verebilir, min 5 karakter
        if len(metin.strip()) < 5:
            # Kısa metinlerde karakter seti kontrolü
            oran = turkce_karakter_orani(metin)
            return "turkish" if oran > 0.5 else "belirsiz"
        sonuc = detector.detect_language_of(metin.strip()[:200])
        return sonuc.iso_code_639_1.name.lower() if sonuc else "belirsiz"
    except Exception:
        return "belirsiz"

print("🌐 Dil tespiti yapılıyor...")
from tqdm import tqdm
tqdm.pandas(desc="Dil tespiti")
veri["_soru_dil"] = veri["_soru"].progress_apply(dili_tespit_et)
veri["_hata_turkce_degil"] = ~veri["_soru_dil"].isin(["turkish", "bos", "belirsiz"])
hata_kolonlari.append("_hata_turkce_degil")

dil_dagilimi = veri["_soru_dil"].value_counts()
print(f"📊 Dil dağılımı:\n{dil_dagilimi}")
print(f"✅ Hücre 5 tamam.")

## 📝 Hücre 6: Yazım Hatası Tespiti (Kategori 9)

Hunspell Türkçe sözlüğü ile yazım hatası oranı hesaplanır.

In [ ]:
# ============================================================
# HÜCRE 6: Yazım hatası kontrolü
# ============================================================
import hunspell

# Hunspell sözlüklerini yükle
try:
    hobj = hunspell.HunSpell("/usr/share/hunspell/tr_TR.dic", "/usr/share/hunspell/tr_TR.aff")
    print("✅ Hunspell Türkçe sözlük yüklendi.")
except Exception:
    # Alternatif yol
    try:
        hobj = hunspell.HunSpell("/usr/share/myspell/tr_TR.dic", "/usr/share/myspell/tr_TR.aff")
        print("✅ Hunspell Türkçe sözlük yüklendi (alternatif yol).")
    except Exception as e:
        print(f"⚠️ Hunspell yüklenemedi: {e}")
        print("   Yazım denetimi atlanacak.")
        hobj = None

YAZIM_HATASI_ESIK = 0.20  # %20'den fazla yazım hatası → şüpheli

def yazim_hatasi_orani(metin: str) -> float:
    """Kelimelerin yüzde kaçı yazım hatası içeriyor?"""
    if not metin or hobj is None:
        return 0.0
    kelimeler = re.findall(r"\b[a-zA-ZçğıöşüÇĞİÖŞÜ]{3,}\b", metin.lower())
    if not kelimeler:
        return 0.0
    hatali = sum(1 for k in kelimeler if not hobj.spell(k))
    return hatali / len(kelimeler)

if hobj is not None:
    print("📝 Yazım denetimi yapılıyor...")
    tqdm.pandas(desc="Yazım denetimi")
    veri["_yazim_hatasi_orani"] = veri["_soru"].progress_apply(yazim_hatasi_orani)
    veri["_hata_yazim"] = veri["_yazim_hatasi_orani"] > YAZIM_HATASI_ESIK
    hata_kolonlari.append("_hata_yazim")
    print(f"   Yazım hatası olan satır: {veri['_hata_yazim'].sum()}")
else:
    veri["_yazim_hatasi_orani"] = 0.0
    veri["_hata_yazim"] = False
    print("   ⚠️ Yazım denetimi atlandı.")

print(f"✅ Hücre 6 tamam.")

## 🧠 Hücre 7: Embedding Modellerini Yükle (GPU)

**A100 için optimize.** İki model yüklenir:
1. `intfloat/multilingual-e5-large-instruct` — Ana çok dilli embedding
2. `dbmdz/bert-base-turkish-cased` — Native Türkçe embedding

⚠️ İlk çalıştırmada modeller indirilir (~2-3 GB), sonraki çalıştırmalarda cache'ten yüklenir.

In [ ]:
# ============================================================
# HÜCRE 7: Embedding modellerini yükle
# ============================================================
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Cihaz: {device}")
if device == "cuda":
    print(f"   GPU Bellek: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"   GPU Adı: {torch.cuda.get_device_name(0)}")

# --- Model 1: multilingual-e5-large-instruct ---
print("📥 Model 1/2 yükleniyor: multilingual-e5-large-instruct (1.1B param)...")
model_e5 = SentenceTransformer(
    "intfloat/multilingual-e5-large-instruct",
    device=device,
    trust_remote_code=False,
)
model_e5.half()  # FP16 → A100'de 2x hız
print(f"✅ Model 1 hazır. Boyut: {model_e5.get_sentence_embedding_dimension()}")

# --- Model 2: bert-base-turkish-cased ---
print("📥 Model 2/2 yükleniyor: bert-base-turkish-cased (768d)...")
model_tr = SentenceTransformer(
    "dbmdz/bert-base-turkish-cased",
    device=device,
    trust_remote_code=False,
)
model_tr.half()
print(f"✅ Model 2 hazır. Boyut: {model_tr.get_sentence_embedding_dimension()}")

print(f"\n✅ Her iki model yüklendi. GPU bellek: {torch.cuda.memory_allocated() / 1e9:.1f} GB kullanılıyor.")

## 🔢 Hücre 8: Tüm Satırlar için Embedding Üret

Her satır için soru embedding'i + cevap embedding'i üretilir (iki model ile).

In [ ]:
# ============================================================
# HÜCRE 8: Embedding üretimi
# ============================================================
import time

# Geçerli soruları al (boş olmayan)
gecerli_mask = veri["_soru"].str.strip() != ""
sorular = veri.loc[gecerli_mask, "_soru"].tolist()
cevaplar = veri.loc[gecerli_mask, "_cevap"].tolist()
gecerli_indeksler = veri[gecerli_mask].index.tolist()

print(f"🔢 Embedding üretilecek satır: {len(sorular)}")

# Batch encoding (GPU optimize)
BATCH_SIZE = 64

def batch_encode(model, metinler, on_ek="", batch_size=BATCH_SIZE):
    """GPU batch encoding with progress."""
    t0 = time.time()
    vektorler = model.encode(
        metinler,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    gecen = time.time() - t0
    print(f"   {on_ek}: {len(metinler)} metin → {vektorler.shape}, {gecen:.1f}s")
    return vektorler

print("\n--- Model 1: e5-large-instruct (sorular) ---")
# e5 modeli özel prefix ister
sorular_e5 = [f"query: {s}" for s in sorular]
soru_vektorleri_e5 = batch_encode(model_e5, sorular_e5, "e5-soru")

print("\n--- Model 1: e5-large-instruct (cevaplar) ---")
cevaplar_e5 = [f"passage: {c}" for c in cevaplar]
cevap_vektorleri_e5 = batch_encode(model_e5, cevaplar_e5, "e5-cevap")

print("\n--- Model 2: bert-turkish (sorular) ---")
soru_vektorleri_tr = batch_encode(model_tr, sorular, "tr-soru")

print("\n--- Model 2: bert-turkish (cevaplar) ---")
cevap_vektorleri_tr = batch_encode(model_tr, cevaplar, "tr-cevap")

# Geçersiz satırlar için NaN doldur
veri["_soru_embed_e5"] = None
veri["_cevap_embed_e5"] = None
veri["_soru_embed_tr"] = None
veri["_cevap_embed_tr"] = None

for i, idx in enumerate(gecerli_indeksler):
    veri.at[idx, "_soru_embed_e5"] = soru_vektorleri_e5[i]
    veri.at[idx, "_cevap_embed_e5"] = cevap_vektorleri_e5[i]
    veri.at[idx, "_soru_embed_tr"] = soru_vektorleri_tr[i]
    veri.at[idx, "_cevap_embed_tr"] = cevap_vektorleri_tr[i]

print(f"\n✅ Embedding üretimi tamamlandı.")
# GPU belleği temizle (cross-encoder için yer aç)
del model_e5
torch.cuda.empty_cache()
print("🧹 Model 1 bellekten kaldırıldı.")

## 🔁 Hücre 9: Yinelenen/Benzer Soru Tespiti (Kategori 1-2)

Cosine similarity matrix hesaplanır. İki modelin de onayladığı (>0.92) eşleşmeler **kesin**, tek modelin bulduğu (>0.85) eşleşmeler **şüpheli** olarak işaretlenir.

In [ ]:
# ============================================================
# HÜCRE 9: Yinelenen/benzer soru tespiti
# ============================================================
from sklearn.metrics.pairwise import cosine_similarity

# Geçerli indeksler için vektörleri numpy array'e dönüştür
s_e5 = np.vstack([veri.at[idx, "_soru_embed_e5"] for idx in gecerli_indeksler])
s_tr = np.vstack([veri.at[idx, "_soru_embed_tr"] for idx in gecerli_indeksler])

print(f"📐 Cosine similarity hesaplanıyor ({len(s_e5)}×{len(s_e5)} matris)...")

# e5 modeli
t0 = time.time()
sim_e5 = cosine_similarity(s_e5)
print(f"   e5 matrix: {time.time()-t0:.1f}s")

# Türkçe model
t0 = time.time()
sim_tr = cosine_similarity(s_tr)
print(f"   tr matrix: {time.time()-t0:.1f}s")

# --- Eşik değerleri ---
BENZERLIK_KESIN_ESIK = 0.92   # İki model de bu eşiğin üstündeyse → kesin hatalı
BENZERLIK_SUPHELI_ESIK = 0.85 # Tek model bu eşiğin üstündeyse → şüpheli

# --- Eşleşmeleri tara ---
# Her satır için: kendisinden önceki satırlarla benzerlik var mı?
veri["_benzer_sira"] = -1
veri["_benzer_skor_e5"] = 0.0
veri["_benzer_skor_tr"] = 0.0
veri["_benzer_kesinlik"] = "temiz"  # temiz / supheli / kesin

n = len(gecerli_indeksler)
print(f"\n🔍 Eşleşmeler taranıyor ({n} satır)...")

kesin_sayisi = 0
supheli_sayisi = 0

for i in range(n):
    idx = gecerli_indeksler[i]
    # Sadece kendinden önceki satırlara bak (üçgen matrisin alt yarısı)
    for j in range(i):
        skor_e5 = sim_e5[i, j]
        skor_tr = sim_tr[i, j]
        
        if skor_e5 > BENZERLIK_KESIN_ESIK and skor_tr > BENZERLIK_KESIN_ESIK:
            # İki model de onaylıyor → kesin yinelenen
            veri.at[idx, "_benzer_sira"] = gecerli_indeksler[j]
            veri.at[idx, "_benzer_skor_e5"] = skor_e5
            veri.at[idx, "_benzer_skor_tr"] = skor_tr
            veri.at[idx, "_benzer_kesinlik"] = "kesin"
            kesin_sayisi += 1
            break
        elif (skor_e5 > BENZERLIK_SUPHELI_ESIK or skor_tr > BENZERLIK_SUPHELI_ESIK):
            # Tek model buldu → şüpheli (ama daha yüksek skorlu bir eşleşme yoksa)
            if veri.at[idx, "_benzer_kesinlik"] != "kesin":
                veri.at[idx, "_benzer_sira"] = gecerli_indeksler[j]
                veri.at[idx, "_benzer_skor_e5"] = skor_e5
                veri.at[idx, "_benzer_skor_tr"] = skor_tr
                veri.at[idx, "_benzer_kesinlik"] = "supheli"
                supheli_sayisi += 1
    
    if (i + 1) % 100 == 0:
        print(f"   İlerleme: {i+1}/{n}")

print(f"\n📊 Sonuç:")
print(f"   Kesin yinelenen: {kesin_sayisi}")
print(f"   Şüpheli benzer: {supheli_sayisi}")
print(f"   Temiz: {n - kesin_sayisi - supheli_sayisi}")

# Hata sütunları
veri["_hata_yinelenen_kesin"] = veri["_benzer_kesinlik"] == "kesin"
veri["_hata_yinelenen_supheli"] = veri["_benzer_kesinlik"] == "supheli"
hata_kolonlari.extend(["_hata_yinelenen_kesin", "_hata_yinelenen_supheli"])

# Bellek temizle
del sim_e5, sim_tr, s_e5, s_tr
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"\n✅ Hücre 9 tamam.")

## 🎯 Hücre 10: Outlier Tespiti (Kategori 3)

DBSCAN + Isolation Forest ile vektör uzayında konu dışı/aykırı kayıtlar tespit edilir.

In [ ]:
# ============================================================
# HÜCRE 10: Outlier tespiti
# ============================================================
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest

# Geçerli embedding'ler
s_e5_out = np.vstack([veri.at[idx, "_soru_embed_e5"] for idx in gecerli_indeksler])
s_tr_out = np.vstack([veri.at[idx, "_soru_embed_tr"] for idx in gecerli_indeksler])

# --- DBSCAN (e5 embedding ile) ---
print("🔍 DBSCAN kümeleme...")
dbscan = DBSCAN(eps=0.3, min_samples=5, metric="cosine", n_jobs=-1)
dbscan_label = dbscan.fit_predict(s_e5_out)
outlier_dbscan = dbscan_label == -1
print(f"   DBSCAN outlier: {outlier_dbscan.sum()} / {len(outlier_dbscan)}")

# --- Isolation Forest (iki model consensus) ---
print("🌲 Isolation Forest...")
iso_e5 = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso_tr = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
outlier_iso_e5 = iso_e5.fit_predict(s_e5_out) == -1
outlier_iso_tr = iso_tr.fit_predict(s_tr_out) == -1
# Consensus: iki model de outlier diyorsa
outlier_iso_consensus = outlier_iso_e5 & outlier_iso_tr

# --- Birleştir ---
veri["_hata_outlier"] = False
for i, idx in enumerate(gecerli_indeksler):
    if outlier_dbscan[i] or outlier_iso_consensus[i]:
        veri.at[idx, "_hata_outlier"] = True

hata_kolonlari.append("_hata_outlier")
print(f"\n📊 Toplam outlier: {veri['_hata_outlier'].sum()}")
print(f"   DBSCAN: {outlier_dbscan.sum()}")
print(f"   IsoForest consensus: {outlier_iso_consensus.sum()}")

del s_e5_out, s_tr_out
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Hücre 10 tamam.")

## 🔗 Hücre 11: Soru-Cevap Semantik Uyum Kontrolü (Kategori 4)

Soru embedding'i ile cevap embedding'i arasındaki cosine similarity ölçülür.

In [ ]:
# ============================================================
# HÜCRE 11: Soru-cevap uyum kontrolü
# ============================================================

QA_UYUMSUZLUK_ESIK = 0.40  # Cosine similarity < 0.4 → uyumsuz
QA_SUPHELI_ESIK = 0.55     # Cosine similarity < 0.55 → şüpheli

print("🔗 Soru-cevap uyumu hesaplanıyor...")

veri["_qa_cosine_e5"] = 0.0
veri["_qa_cosine_tr"] = 0.0
veri["_hata_qa_uyumsuz"] = False
veri["_hata_qa_supheli"] = False

uyumsuz = 0
supheli = 0

for idx in gecerli_indeksler:
    s_e5_vec = veri.at[idx, "_soru_embed_e5"]
    c_e5_vec = veri.at[idx, "_cevap_embed_e5"]
    s_tr_vec = veri.at[idx, "_soru_embed_tr"]
    c_tr_vec = veri.at[idx, "_cevap_embed_tr"]
    
    # Cosine similarity = dot product (zaten normalize)
    c_e5 = float(np.dot(s_e5_vec, c_e5_vec))
    c_tr = float(np.dot(s_tr_vec, c_tr_vec))
    
    veri.at[idx, "_qa_cosine_e5"] = c_e5
    veri.at[idx, "_qa_cosine_tr"] = c_tr
    
    if c_e5 < QA_UYUMSUZLUK_ESIK and c_tr < QA_UYUMSUZLUK_ESIK:
        veri.at[idx, "_hata_qa_uyumsuz"] = True
        uyumsuz += 1
    elif c_e5 < QA_SUPHELI_ESIK or c_tr < QA_SUPHELI_ESIK:
        veri.at[idx, "_hata_qa_supheli"] = True
        supheli += 1

hata_kolonlari.extend(["_hata_qa_uyumsuz", "_hata_qa_supheli"])

print(f"📊 Soru-cevap uyum sonuçları:")
print(f"   Kesin uyumsuz: {uyumsuz}")
print(f"   Şüpheli: {supheli}")
print(f"   Temiz: {len(gecerli_indeksler) - uyumsuz - supheli}")

# Ortalama cosine değerlerini göster
gecerli_qa_e5 = veri.loc[gecerli_mask, "_qa_cosine_e5"]
gecerli_qa_tr = veri.loc[gecerli_mask, "_qa_cosine_tr"]
print(f"   e5 ortalama QA cosine: {gecerli_qa_e5.mean():.4f}")
print(f"   tr ortalama QA cosine: {gecerli_qa_tr.mean():.4f}")

print(f"✅ Hücre 11 tamam.")

## 🏷️ Hücre 12: Kategori-Modül Uyum Kontrolü (Kategori 11)

Kategori embedding'i ile soru embedding'i arasındaki uzaklık ölçülür.

In [ ]:
# ============================================================
# HÜCRE 12: Kategori-modül uyum kontrolü
# ============================================================

# Önce her benzersiz kategori için embedding oluştur
benzersiz_kategoriler = veri["_kategori"].unique()
print(f"🏷️ Benzersiz kategori: {len(benzersiz_kategoriler)}")

# Kategori embedding'leri (sadece model_tr kullan)
kategori_metinleri = [f"Kategori: {k}" for k in benzersiz_kategoriler]
kategori_embed = model_tr.encode(
    kategori_metinleri,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
kategori_embed_haritasi = {k: kategori_embed[i] for i, k in enumerate(benzersiz_kategoriler)}

# Her satır için soru-kategori benzerliği
KATEGORI_UYUMSUZLUK_ESIK = 0.30

veri["_kategori_cosine"] = 0.0
veri["_hata_kategori_uyumsuz"] = False

uyumsuz_sayi = 0
for idx in gecerli_indeksler:
    kategori = veri.at[idx, "_kategori"]
    s_vec = veri.at[idx, "_soru_embed_tr"]
    k_vec = kategori_embed_haritasi.get(kategori)
    if k_vec is not None:
        cos = float(np.dot(s_vec, k_vec))
        veri.at[idx, "_kategori_cosine"] = cos
        if cos < KATEGORI_UYUMSUZLUK_ESIK:
            veri.at[idx, "_hata_kategori_uyumsuz"] = True
            uyumsuz_sayi += 1

hata_kolonlari.append("_hata_kategori_uyumsuz")
print(f"📊 Kategori uyumsuz: {uyumsuz_sayi}")
print(f"✅ Hücre 12 tamam.")

# Model 2'yi de temizle
del model_tr
torch.cuda.empty_cache()

## 📋 Hücre 13: Tüm Hataları Birleştir & Sınıflandır

In [ ]:
# ============================================================
# HÜCRE 13: Hata konsolidasyonu
# ============================================================

HATA_SINIFI_ETIKETLERI = {
    "_hata_bos_kategori": ("Kategorisi boş", "DÜZELT"),
    "_hata_bos_soru": ("Sorusu boş", "SİL"),
    "_hata_bos_cevap": ("Cevabı boş", "SİL"),
    "_hata_kisa_soru": ("Çok kısa soru (<3 kelime)", "İNCELE"),
    "_hata_uzun_soru": ("Çok uzun soru (>150 kelime)", "İNCELE"),
    "_hata_sadece_sembol": ("Sadece sayı/sembol", "SİL"),
    "_hata_dusuk_bilgi": ("Bilgi yoğunluğu düşük", "İNCELE"),
    "_hata_turkce_degil": ("Türkçe değil", "İNCELE"),
    "_hata_yazim": ("Yazım hatası yoğun", "İNCELE"),
    "_hata_yinelenen_kesin": ("Kesin yinelenen soru", "SİL"),
    "_hata_yinelenen_supheli": ("Şüpheli benzer soru", "İNCELE"),
    "_hata_outlier": ("Konu dışı/aykırı kayıt", "İNCELE"),
    "_hata_qa_uyumsuz": ("Soru-cevap uyumsuz", "SİL"),
    "_hata_qa_supheli": ("Soru-cevap şüpheli", "İNCELE"),
    "_hata_kategori_uyumsuz": ("Kategori uyumsuz", "İNCELE"),
}

# Her satır için hata sınıflarını topla
veri["_hata_siniflari"] = ""
veri["_oneri_aksiyon"] = "TEMİZ"

for idx in veri.index:
    siniflar = []
    en_kotu_aksiyon = "TEMİZ"
    aksiyon_onceligi = {"SİL": 3, "DÜZELT": 2, "İNCELE": 1, "TEMİZ": 0}
    
    for kolon, (etiket, aksiyon) in HATA_SINIFI_ETIKETLERI.items():
        if kolon in veri.columns and veri.at[idx, kolon]:
            siniflar.append(etiket)
            if aksiyon_onceligi[aksiyon] > aksiyon_onceligi[en_kotu_aksiyon]:
                en_kotu_aksiyon = aksiyon
    
    veri.at[idx, "_hata_siniflari"] = "; ".join(siniflar) if siniflar else ""
    veri.at[idx, "_oneri_aksiyon"] = en_kotu_aksiyon

# İstatistikler
print("=" * 60)
print("📊 HATA KONSOLİDASYON RAPORU")
print("=" * 60)
for kolon, (etiket, _) in HATA_SINIFI_ETIKETLERI.items():
    if kolon in veri.columns:
        sayi = veri[kolon].sum()
        if sayi > 0:
            print(f"  {etiket}: {sayi}")

print(f"\n📊 Aksiyon dağılımı:")
aksiyon_dagilimi = veri["_oneri_aksiyon"].value_counts()
for aksiyon, sayi in aksiyon_dagilimi.items():
    print(f"  {aksiyon}: {sayi}")

print(f"\n📊 Toplam satır: {len(veri)}")
hatali_satir = (veri["_oneri_aksiyon"] != "TEMİZ").sum()
temiz_satir = (veri["_oneri_aksiyon"] == "TEMİZ").sum()
print(f"  Hatalı/Şüpheli: {hatali_satir} ({hatali_satir/len(veri)*100:.1f}%)")
print(f"  Temiz: {temiz_satir} ({temiz_satir/len(veri)*100:.1f}%)")
print(f"\n✅ Hücre 13 tamam.")

## 📤 Hücre 14: Çıktıları Üret

**İki Excel dosyası + bir JSON özet oluşturulur:**
1. `temiz_veri.xlsx` — Orijinal sütunlar + sadece TEMİZ aksiyonlu satırlar
2. `hatali_kayitlar.xlsx` — Tüm satırlar + hata sınıfları + aksiyon
3. `ozet.json` — İstatistiksel özet

In [ ]:
# ============================================================
# HÜCRE 14: Çıktı dosyalarını oluştur
# ============================================================

# --- Orijinal sütunları koru ---
orijinal_kolonlar = list(veri.columns)
# Yardımcı sütunları çıkar (özel sütunlar)
yardimci_kolonlar = [k for k in veri.columns if k.startswith("_")]

# --- Temiz veri ---
temiz_veri = veri[veri["_oneri_aksiyon"] == "TEMİZ"].copy()
# Orijinal sütunları + birkaç faydalı sütunu ekle
temiz_cikti_kolonlari = [k for k in orijinal_kolonlar if not k.startswith("_")]
# Embedding sütunlarını çıkar (çok büyük)
temiz_cikti_kolonlari = [k for k in temiz_cikti_kolonlari if "embed" not in k.lower()]
temiz_cikti = temiz_veri[temiz_cikti_kolonlari] if all(k in temiz_veri.columns for k in temiz_cikti_kolonlari) else temiz_veri

temiz_yolu = os.path.join(CIKTI_DIZINI, "temiz_veri.xlsx")
temiz_cikti.to_excel(temiz_yolu, index=False, engine="openpyxl")
print(f"✅ Temiz veri kaydedildi: {temiz_yolu}")
print(f"   {len(temiz_cikti)} satır")

# --- Hatalı kayıtlar raporu ---
rapor_kolonlari = [k for k in orijinal_kolonlar if not k.startswith("_") and "embed" not in k.lower()]
# Hata bilgilerini ekle
ek_kolonlar = ["_hata_siniflari", "_oneri_aksiyon"]
# Mevcutsa sayısal metrikleri de ekle
for ek in ["_benzer_kesinlik", "_benzer_skor_e5", "_benzer_skor_tr", "_qa_cosine_e5", "_qa_cosine_tr",
           "_soru_dil", "_yazim_hatasi_orani", "_soru_token", "_soru_ttr", "_kategori_cosine"]:
    if ek in veri.columns:
        ek_kolonlar.append(ek)

rapor_kolonlari = [k for k in rapor_kolonlari if k in veri.columns]
rapor_veri = veri[rapor_kolonlari + ek_kolonlar].copy()

# İnsan tarafından okunabilir sütun isimleri
rapor_veri.columns = [c.replace("_", " ").strip().title() for c in rapor_veri.columns]

rapor_yolu = os.path.join(CIKTI_DIZINI, "hatali_kayitlar.xlsx")
rapor_veri.to_excel(rapor_yolu, index=False, engine="openpyxl")
print(f"✅ Hatalı kayıt raporu kaydedildi: {rapor_yolu}")
print(f"   {len(rapor_veri)} satır (tüm satırlar, hata sınıflarıyla)")

# --- JSON özet ---
ozet = {
    "olusturma_tarihi": datetime.now().isoformat(),
    "kaynak_dosya": os.path.basename(EXCEL_DOSYASI),
    "toplam_satir": int(len(veri)),
    "temiz_satir": int(temiz_satir),
    "hatali_satir": int(hatali_satir),
    "aksiyon_dagilimi": {k: int(v) for k, v in aksiyon_dagilimi.items()},
    "hata_dagilimi": {},
}
for kolon, (etiket, _) in HATA_SINIFI_ETIKETLERI.items():
    if kolon in veri.columns:
        sayi = int(veri[kolon].sum())
        if sayi > 0:
            ozet["hata_dagilimi"][etiket] = sayi

ozet_yolu = os.path.join(CIKTI_DIZINI, "ozet.json")
with open(ozet_yolu, "w", encoding="utf-8") as f:
    json.dump(ozet, f, ensure_ascii=False, indent=2)
print(f"✅ Özet kaydedildi: {ozet_yolu}")

print(f"\n{'='*60}")
print(f"📁 Tüm çıktılar: {CIKTI_DIZINI}/")
print(f"   📄 temiz_veri.xlsx     → {len(temiz_cikti)} satır temiz veri")
print(f"   📄 hatali_kayitlar.xlsx → Tüm satırlar + hata sınıfları")
print(f"   📄 ozet.json            → İstatistiksel özet")
print(f"{'='*60}")
print(f"\n🎉 İŞLEM TAMAMLANDI!")
print(f"\n📌 Sonraki adım:")
print(f"   1. 'hatali_kayitlar.xlsx' dosyasını inceleyin.")
print(f"   2. 'İNCELE' aksiyonlu satırları manuel değerlendirin.")
print(f"   3. Onayladığınız satırları 'temiz_veri.xlsx'e ekleyin.")
print(f"   4. Temiz veriyi HadesBot'a aktarın:")
print(f"      python excel_ice_aktar.py ciktilar/temiz_veri.xlsx")
print(f"      python vektorleri_olustur.py")

---
## 📝 Notlar

- **Eşik değerleri** ihtiyaca göre ayarlanabilir (Hücre 9, 10, 11, 12'deki `_ESIK` değişkenleri).
- **Şüpheli kayıtlar** manuel inceleme için işaretlenir, otomatik silinmez.
- **A100 GPU** kullanıldığında tüm pipeline ~2-3 dakika sürer.
- Excel dosyası 5000+ satırsa, batch_size'ı 32'ye düşürebilirsiniz.

---
**HadesBot · UYDS SSS Veri Ayıklama · v1.0**